In [17]:
!apt-get install -y glpk-utils -q

/bin/bash: apt-get: command not found


# Modelo MILP — Asignación de Horarios Académicos
## Universidad Externado de Colombia · Departamento de Matemáticas

Este notebook implementa un **Modelo de Programación Lineal Entera Mixta (MILP)**  
para optimizar la asignación de horarios académicos.

### Estructura general del modelo

Un modelo MILP tiene tres elementos:

| Elemento | Descripción | Ejemplo en este modelo |
|---|---|---|
| **Variables de decisión** | ¿Qué decide el solver? | `x_t[m,g,d,t]` = ¿asignar grupo g de materia m el día d a la hora t en salón teórico? |
| **Función objetivo** | ¿Qué queremos maximizar/minimizar? | Maximizar sesiones asignadas, penalizar holguras |
| **Restricciones** | ¿Qué reglas debe respetar? | Capacidad de salones, días permitidos, una clase por día |

### Restricciones: duras vs. blandas

- **Restricciones DURAS** (`HC` — Hard Constraints): deben cumplirse **siempre**.  
  Si no se cumplen, el horario es inválido.  
  _Ejemplo: no poner dos grupos en el mismo salón a la misma hora._

- **Restricciones BLANDAS** (`SC` — Soft Constraints): deseables pero no obligatorias.  
  Se modelan como **penalizaciones** en la función objetivo.  
  _Ejemplo: todas las sesiones de un grupo en la misma franja horaria._

### Cómo funciona activar/desactivar restricciones

En Pyomo, `model.restriccion.deactivate()` le dice al solver que **ignore** esa restricción.  
Se usa para:
- Diagnosticar qué restricción está causando infactibilidad
- Relajar temporalmente una restricción para obtener al menos una solución parcial


## 1. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
from pyomo.environ import *
from pyomo.opt import SolverStatus, TerminationCondition


Librerías cargadas correctamente


## 2. Lectura de datos

Los datos de entrada provienen de un Excel con las siguientes columnas:
- `nombre_materia`: nombre de la asignatura
- `numero_sesiones`: total de sesiones semanales requeridas (2 = LMi o MaJ; 3 = LMiVi)
- `Cantidad_sesiones_computo`: cuántas de esas sesiones requieren sala de cómputo
- `No_grupos`: número de grupos que se ofrecen para esa materia


In [6]:
df_proyecciones = pd.read_excel("data_set_proyeccion_salones.xlsx")
print("Materias cargadas:", len(df_proyecciones))
df_proyecciones.head()

Materias cargadas: 43


,Semestre,nombre_materia,Cantidad_sesiones_computo,numero_sesiones,Estudiantes_Actuales,Inscritos_Nuevos,Perdida_%,Aprobaron,Perdieron,Prerrequisitos,Prerrequisito_Inmediato,Prerrequisitos_Aprobados,Estudiantes_Proyectados,Capacidad_usada,No_grupos,division_grupos
0,1,Cálculo 1,0,3,58,50,17.66,48,10,Ninguno,Ninguno,0,60,20,3,"[20, 20, 20]"
1,1,Idioma 1,0,2,96,50,8.22,88,8,Ninguno,Ninguno,0,58,30,2,"[29, 29]"
2,1,Instituciones Políticas,0,2,98,50,2.13,96,2,Ninguno,Ninguno,0,52,30,2,"[26, 26]"
3,1,Introducción a la Ciencia de Datos,2,2,104,50,9.26,94,10,Ninguno,Ninguno,0,60,30,2,"[30, 30]"
4,1,Matemáticas Discretas,0,3,113,50,19.36,91,22,Ninguno,Ninguno,0,72,20,4,"[18, 18, 18, 18]"


## 3. Disponibilidad de salones y franjas

Define cuántos salones hay disponibles por tipo (teórico / cómputo) para cada día.  
Esta es la **capacidad** del sistema — limita cuántas clases pueden ocurrir en paralelo.


In [53]:
# Número de salones disponibles por día
# slots_teorico: cuántos salones teóricos (aulas normales) están disponibles
# slots_computo: cuántas salas de cómputo están disponibles

"""""
disponibilidad_df = pd.DataFrame({
    "dia":           ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes"],
    "slots_teorico": [4,       4,        3,           2,        2        ],
    "slots_computo": [2,       2,        2,           2,        2       ]
})
"""""

disponibilidad_df = pd.DataFrame({
    "dia":           ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes"],
    "slots_teorico": [7,       7,        7,           7,        7        ],
    "slots_computo": [5,       2,        5,           2,        5        ]
})


dias  = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes"]
horas = ["7-9", "9-11", "11-13", "14-16", "16-18"]  # franjas de 2 horas

print("Disponibilidad de salones:")
print(disponibilidad_df.to_string(index=False))
print(f"\nFranjas horarias disponibles: {horas}")


Disponibilidad de salones:
      dia  slots_teorico  slots_computo
    Lunes              7              5
   Martes              7              2
Miercoles              7              5
   Jueves              7              2
  Viernes              7              5

Franjas horarias disponibles: ['7-9', '9-11', '11-13', '14-16', '16-18']


## 4. Diagnóstico de datos de entrada

Antes de construir el modelo, verificamos que los datos sean consistentes.  
Si hay errores aquí, el solver fallará o dará resultados incorrectos.


# #1

In [54]:
print("=" * 60)
print("DIAGNÓSTICO DE DATOS DE ENTRADA")
print("=" * 60)

errores  = []
warnings = []

for _, row in df_proyecciones.iterrows():
    m  = row["nombre_materia"]
    ns = int(row["numero_sesiones"])
    nc = int(row["Cantidad_sesiones_computo"])
    ng = int(row["No_grupos"])

    # Error crítico: más sesiones de cómputo que sesiones totales
    if nc > ns:
        errores.append(
            f"'{m}': sesiones_computo ({nc}) > numero_sesiones ({ns})"
        )

    # Error crítico: grupos inválidos
    if ng <= 0:
        errores.append(f"'{m}': No_grupos debe ser ≥ 1, tiene ({ng})")

    # Error crítico: sesiones fuera del rango válido (1 o 2 o 3)
    if ns not in [1, 2, 3]:
        errores.append(
            f"'{m}': numero_sesiones debe ser 1, 2 o 3 — tiene ({ns})"
        )

    # Warning: materia mixta con pocas franjas para separar cómputo y teórico
    if 0 < nc < ns and ns == 2:
        warnings.append(
            f"'{m}': materia mixta con 2 sesiones — "
            "se asignarán días distintos para teórico y cómputo"
        )

if errores:
    print("\n ERRORES — Corrige antes de ejecutar el solver:\n")
    for e in errores:
        print(f"   {e}")
else:
    print("Sin errores críticos")

if warnings:
    print("\nADVERTENCIAS (no bloquean el solver):\n")
    for w in warnings:
        print(f"   {w}")

# Diagnóstico de capacidad: ¿hay suficientes salones para todas las clases?
print("\n" + "=" * 60)
print("DIAGNÓSTICO DE CAPACIDAD")
print("=" * 60)

total_grupos = sum(
    int(row["No_grupos"]) for _, row in df_proyecciones.iterrows()
)
max_slots_teo  = max(disponibilidad_df["slots_teorico"]) * len(horas)
max_slots_comp = max(disponibilidad_df["slots_computo"]) * len(horas)

total_ses_comp = sum(
    int(row["Cantidad_sesiones_computo"]) * int(row["No_grupos"])
    for _, row in df_proyecciones.iterrows()
)
total_ses_teo = sum(
    (int(row["numero_sesiones"]) - int(row["Cantidad_sesiones_computo"]))
    * int(row["No_grupos"])
    for _, row in df_proyecciones.iterrows()
)

print(f"   Grupos totales a programar:      {total_grupos}")
print(f"   Sesiones teóricas a asignar:     {total_ses_teo}")
print(f"   Sesiones de cómputo a asignar:   {total_ses_comp}")
print(f"   Slots teóricos disponibles (max/día × franjas × días): "
      f"{max(disponibilidad_df['slots_teorico'])} × {len(horas)} × {len(dias)} "
      f"= {max(disponibilidad_df['slots_teorico']) * len(horas) * len(dias)}")
print(f"   Slots cómputo disponibles:       "
      f"{max(disponibilidad_df['slots_computo'])} × {len(horas)} × {len(dias)} "
      f"= {max(disponibilidad_df['slots_computo']) * len(horas) * len(dias)}")

if total_ses_comp > max(disponibilidad_df["slots_computo"]) * len(horas) * len(dias):
    print("\n ALERTA: Las sesiones de cómputo SUPERAN la capacidad total disponible.")
    print("   → Considera reducir grupos, agregar salas o ampliar franjas.")
else:
    print("\n La capacidad teórica es suficiente para acomodar todas las sesiones.")


DIAGNÓSTICO DE DATOS DE ENTRADA
Sin errores críticos

ADVERTENCIAS (no bloquean el solver):

   'Probabilidad': materia mixta con 2 sesiones — se asignarán días distintos para teórico y cómputo
   'Métodos Numéricos': materia mixta con 2 sesiones — se asignarán días distintos para teórico y cómputo
   'Ciberseguridad': materia mixta con 2 sesiones — se asignarán días distintos para teórico y cómputo

DIAGNÓSTICO DE CAPACIDAD
   Grupos totales a programar:      119
   Sesiones teóricas a asignar:     175
   Sesiones de cómputo a asignar:   99
   Slots teóricos disponibles (max/día × franjas × días): 7 × 5 × 5 = 175
   Slots cómputo disponibles:       5 × 5 × 5 = 125

 La capacidad teórica es suficiente para acomodar todas las sesiones.


# #2

In [65]:
print("=" * 60)
print("DIAGNÓSTICO DE DATOS DE ENTRADA")
print("=" * 60)

errores  = []
warnings = []

for _, row in df_proyecciones.iterrows():
    m  = row["nombre_materia"]
    ns = int(row["numero_sesiones"])
    nc = int(row["Cantidad_sesiones_computo"])
    ng = int(row["No_grupos"])

    if nc > ns:
        errores.append(
            f"'{m}': sesiones_computo ({nc}) > numero_sesiones ({ns})"
        )
    if ng <= 0:
        errores.append(f"'{m}': No_grupos debe ser ≥ 1, tiene ({ng})")
    if ns not in [1, 2, 3]:
        errores.append(
            f"'{m}': numero_sesiones debe ser 1, 2 o 3 — tiene ({ns})"
        )
    if 0 < nc < ns and ns == 2:
        warnings.append(
            f"'{m}': materia mixta con 2 sesiones — "
            "se asignarán días distintos para teórico y cómputo"
        )

if errores:
    print("\nERRORES — Corrige antes de ejecutar el solver:\n")
    for e in errores:
        print(f"   {e}")
else:
    print("Sin errores críticos")

if warnings:
    print("\nADVERTENCIAS (no bloquean el solver):\n")
    for w in warnings:
        print(f"   {w}")

# ── Diagnóstico de capacidad ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("DIAGNÓSTICO DE CAPACIDAD")
print("=" * 60)

# Total de sesiones = sesiones_por_materia × grupos, sumado para todas las materias
total_ses_comp = sum(
    int(row["Cantidad_sesiones_computo"]) * int(row["No_grupos"])
    for _, row in df_proyecciones.iterrows()
)
total_ses_teo = sum(
    (int(row["numero_sesiones"]) - int(row["Cantidad_sesiones_computo"]))
    * int(row["No_grupos"])
    for _, row in df_proyecciones.iterrows()
)
total_sesiones = total_ses_teo + total_ses_comp

# Total de grupos = suma de No_grupos por materia
total_grupos = sum(
    int(row["No_grupos"]) for _, row in df_proyecciones.iterrows()
)

# Slots totales disponibles: suma real día a día × franjas
slots_teo_total  = sum(disponibilidad_df["slots_teorico"]) * len(horas)
slots_comp_total = sum(disponibilidad_df["slots_computo"]) * len(horas)

suma_teo  = "+".join(str(x) for x in disponibilidad_df["slots_teorico"])
suma_comp = "+".join(str(x) for x in disponibilidad_df["slots_computo"])

print(f"\n   DEMANDA")
print(f"   {'─'*40}")
print(f"   Materias a programar:            {len(materias)}")
print(f"   Grupos totales:                  {total_grupos}")
print(f"   Sesiones totales a asignar:      {total_sesiones}")
print(f"      → Teóricas:                   {total_ses_teo}  "
      f"(sesiones_teorico × grupos, sumado por materia)")
print(f"      → Cómputo:                    {total_ses_comp}  "
      f"(sesiones_computo × grupos, sumado por materia)")

print(f"\n   OFERTA (capacidad real por día)")
print(f"   {'─'*40}")
print(f"   Slots teóricos disponibles:      {slots_teo_total}  "
      f"(({suma_teo}) × {len(horas)} franjas)")
print(f"   Slots cómputo disponibles:       {slots_comp_total}  "
      f"(({suma_comp}) × {len(horas)} franjas)")

print(f"\n   BALANCE")
print(f"   {'─'*40}")
balance_teo  = slots_teo_total  - total_ses_teo
balance_comp = slots_comp_total - total_ses_comp

print(f"   Teórico:  {slots_teo_total} disponibles − {total_ses_teo} requeridas = "
      f"{balance_teo:+d}  {'' if balance_teo >= 0 else ' DÉFICIT'}")
print(f"   Cómputo:  {slots_comp_total} disponibles − {total_ses_comp} requeridas = "
      f"{balance_comp:+d}  {'' if balance_comp >= 0 else ' DÉFICIT'}")

if balance_comp < 0 or balance_teo < 0:
    print(f"\n    ALERTA: hay déficit real — el solver no podrá asignar todas las sesiones.")
    print(f"      → Considera aumentar salas o reducir grupos en el Excel.")
else:
    print(f"\n     NOTA: balance positivo no garantiza asignación completa.")
    print(f"      Los días permitidos (MaJ / LMiVi) concentran la demanda")
    print(f"      en subconjuntos de días, reduciendo los slots realmente accesibles.")

DIAGNÓSTICO DE DATOS DE ENTRADA
Sin errores críticos

ADVERTENCIAS (no bloquean el solver):

   'Probabilidad': materia mixta con 2 sesiones — se asignarán días distintos para teórico y cómputo
   'Métodos Numéricos': materia mixta con 2 sesiones — se asignarán días distintos para teórico y cómputo
   'Ciberseguridad': materia mixta con 2 sesiones — se asignarán días distintos para teórico y cómputo

DIAGNÓSTICO DE CAPACIDAD

   DEMANDA
   ────────────────────────────────────────
   Materias a programar:            43
   Grupos totales:                  119
   Sesiones totales a asignar:      274
      → Teóricas:                   175  (sesiones_teorico × grupos, sumado por materia)
      → Cómputo:                    99  (sesiones_computo × grupos, sumado por materia)

   OFERTA (capacidad real por día)
   ────────────────────────────────────────
   Slots teóricos disponibles:      175  ((7+7+7+7+7) × 5 franjas)
   Slots cómputo disponibles:       95  ((5+2+5+2+5) × 5 franjas)

   BA

## 5. Preprocesamiento

Construimos los diccionarios que el modelo usará para indexar variables y restricciones.

### Regla de días permitidos

El número de sesiones determina en qué días puede dictarse la materia:
- `numero_sesiones = 2` → **Martes y Jueves** (2 días en la semana)
- `numero_sesiones = 3` → **Lunes, Miércoles y Viernes** (3 días alternos)
- `numero_sesiones = 1` → **cualquier día** (sesión única)

Esta es una **restricción dura**: no se puede violar el esquema de días.


In [55]:
materias           = df_proyecciones["nombre_materia"].tolist()
grupos_por_materia = {}
sesiones_materia   = {}
sesiones_computo   = {}
sesiones_teorico   = {}
tipo_sesion        = {}

for _, row in df_proyecciones.iterrows():
    m  = row["nombre_materia"]
    ns = int(row["numero_sesiones"])
    nc = int(row["Cantidad_sesiones_computo"])
    nt = ns - nc

    grupos_por_materia[m] = list(range(int(row["No_grupos"])))
    sesiones_materia[m]   = ns
    sesiones_computo[m]   = nc
    sesiones_teorico[m]   = nt

    # Clasifica la materia por tipo de sala que necesita
    if   nc == ns: tipo_sesion[m] = "computo"   # 100% en sala de cómputo
    elif nc == 0:  tipo_sesion[m] = "teorico"   # 100% en aula teórica
    else:          tipo_sesion[m] = "mixta"     # parte teórico, parte cómputo

slots_teorico = dict(zip(disponibilidad_df["dia"],
                         disponibilidad_df["slots_teorico"].astype(int)))
slots_computo = dict(zip(disponibilidad_df["dia"],
                         disponibilidad_df["slots_computo"].astype(int)))

def get_dias(ns):
    """Retorna los días permitidos según el número de sesiones semanales."""
    if ns == 3:
        return ["Lunes", "Miercoles", "Viernes"]
    elif ns == 2:
        return ["Martes", "Jueves"]
    else:  # 1 sesión → cualquier día
        return dias

dias_permitidos = {m: get_dias(sesiones_materia[m]) for m in materias}

# Índices compuestos (materia, grupo)
MG      = [(m, g) for m in materias for g in grupos_por_materia[m]]
MG_teo  = [(m, g) for (m, g) in MG if sesiones_teorico[m]  > 0]
MG_comp = [(m, g) for (m, g) in MG if sesiones_computo[m] > 0]

print(f"Pares (materia, grupo) totales:         {len(MG)}")
print(f"Pares que necesitan salón teórico:      {len(MG_teo)}")
print(f"Pares que necesitan sala de cómputo:    {len(MG_comp)}")
print("\nTipos de materia:")
for t in ["teorico", "computo", "mixta"]:
    n = sum(1 for m in materias if tipo_sesion[m] == t)
    print(f"   {t:10s}: {n} materia(s)")


Pares (materia, grupo) totales:         119
Pares que necesitan salón teórico:      95
Pares que necesitan sala de cómputo:    49

Tipos de materia:
   teorico   : 26 materia(s)
   computo   : 9 materia(s)
   mixta     : 8 materia(s)


## 6. Construcción del modelo MILP

### Variables de decisión

| Variable | Tipo | Significado |
|---|---|---|
| `x_t[m,g,d,t]` | Binaria (0/1) | = 1 si el grupo `g` de la materia `m` tiene sesión **teórica** el día `d` a la hora `t` |
| `x_c[m,g,d,t]` | Binaria (0/1) | = 1 si el grupo `g` de la materia `m` tiene sesión de **cómputo** el día `d` a la hora `t` |
| `z[m,g,t]`     | Binaria (0/1) | = 1 si la franja `t` es **la franja fija** elegida para todas las sesiones de ese grupo _(restricción blanda)_ |
| `holgura_teo[m,g]`  | Real ≥ 0 | Sesiones teóricas que **no se pudieron asignar** (penalización) |
| `holgura_comp[m,g]` | Real ≥ 0 | Sesiones de cómputo que **no se pudieron asignar** (penalización) |

Las variables de **holgura** son la clave para que el modelo no sea infactible:  
en lugar de rechazar una solución porque faltan salones, el solver asigna lo que puede  
y la holgura captura lo que quedó sin asignar (que luego se penaliza en el objetivo).


In [ ]:
model = ConcreteModel()

# ── Conjuntos ─────────────────────────────────────────────────────────────────
model.MG = Set(initialize=MG, dimen=2)
model.D  = Set(initialize=dias)
model.T  = Set(initialize=horas)

# ── Variables de decisión ─────────────────────────────────────────────────────
model.x_t = Var(model.MG, model.D, model.T, domain=Binary)   # sesión teórica
model.x_c = Var(model.MG, model.D, model.T, domain=Binary)   # sesión de cómputo
model.z   = Var(model.MG, model.T, domain=Binary)             # franja fija

model.holgura_teo  = Var(model.MG, domain=NonNegativeReals)   # sesiones teo no asignadas
model.holgura_comp = Var(model.MG, domain=NonNegativeReals)   # sesiones comp no asignadas

print("Variables creadas")
print(f"   x_t: {len(MG)} × {len(dias)} × {len(horas)} = {len(MG)*len(dias)*len(horas)} variables binarias")
print(f"   x_c: {len(MG)} × {len(dias)} × {len(horas)} = {len(MG)*len(dias)*len(horas)} variables binarias")
print(f"   z  : {len(MG)} × {len(horas)} = {len(MG)*len(horas)} variables binarias")
print(f"   holguras: {len(MG)*2} variables continuas")


Variables creadas
   x_t: 119 × 5 × 5 = 2975 variables binarias
   x_c: 119 × 5 × 5 = 2975 variables binarias
   z  : 119 × 5 = 595 variables binarias
   holguras: 238 variables continuas


## 7. Restricciones (Constraints — HC)

Estas restricciones **nunca se desactivan** en producción.  
Son las que hacen que el horario sea físicamente posible.


In [57]:
# ─────────────────────────────────────────────────────────────────────────────
# HC-1: Cumplir el número requerido de sesiones teóricas
#        (con holgura para no causar infactibilidad)
#
#   Σ x_t[m,g,d,t] + holgura_teo[m,g] = sesiones_teorico[m]
#   ∀ (m,g)
#
# Si holgura_teo > 0 → al grupo le faltaron sesiones teóricas (penalizado).
# ─────────────────────────────────────────────────────────────────────────────
def sesiones_teo_rule(model, m, g):
    return (
        sum(model.x_t[(m,g), d, t] for d in model.D for t in model.T)
        + model.holgura_teo[(m,g)]
        == sesiones_teorico[m]
    )
model.HC1_ses_teo = Constraint(model.MG, rule=sesiones_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-2: Cumplir el número requerido de sesiones de cómputo (con holgura)
#
#   Σ x_c[m,g,d,t] + holgura_comp[m,g] = sesiones_computo[m]
#   ∀ (m,g)
# ─────────────────────────────────────────────────────────────────────────────
def sesiones_comp_rule(model, m, g):
    return (
        sum(model.x_c[(m,g), d, t] for d in model.D for t in model.T)
        + model.holgura_comp[(m,g)]
        == sesiones_computo[m]
    )
model.HC2_ses_comp = Constraint(model.MG, rule=sesiones_comp_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-3: Capacidad de salones teóricos por (día, hora)
#
#   Σ x_t[m,g,d,t] ≤ slots_teorico[d]    ∀ d, t
#
# No puede haber más clases simultáneas que salones disponibles.
# ─────────────────────────────────────────────────────────────────────────────
def cap_teo_rule(model, d, t):
    return (
        sum(model.x_t[(m,g), d, t] for (m,g) in MG_teo)
        <= slots_teorico[d]
    )
model.HC3_cap_teo = Constraint(model.D, model.T, rule=cap_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-4: Capacidad de salas de cómputo por (día, hora)
#
#   Σ x_c[m,g,d,t] ≤ slots_computo[d]    ∀ d, t
# ─────────────────────────────────────────────────────────────────────────────
def cap_comp_rule(model, d, t):
    return (
        sum(model.x_c[(m,g), d, t] for (m,g) in MG_comp)
        <= slots_computo[d]
    )
model.HC4_cap_comp = Constraint(model.D, model.T, rule=cap_comp_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-5: Un grupo tiene máximo UNA sesión por día 
#   Σ_t x_t[m,g,d,t] + Σ_t x_c[m,g,d,t] ≤ 1    ∀ (m,g), d
#
# Evita que un grupo tenga dos clases de la misma materia el mismo día.
# ─────────────────────────────────────────────────────────────────────────────
def una_clase_dia_rule(model, m, g, d):
    return (
        sum(model.x_t[(m,g), d, t] for t in model.T) +
        sum(model.x_c[(m,g), d, t] for t in model.T)
    ) <= 1
model.HC5_una_clase_dia = Constraint(model.MG, model.D, rule=una_clase_dia_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-6: Solo asignar en días PERMITIDOS — sesiones teóricas
#
#   x_t[m,g,d,t] = 0    si d ∉ dias_permitidos[m]
#
# Materia con 2 sesiones → solo Martes/Jueves.
# Materia con 3 sesiones → solo Lunes/Miércoles/Viernes.
# ─────────────────────────────────────────────────────────────────────────────
def dias_validos_teo_rule(model, m, g, d, t):
    if d not in dias_permitidos[m]:
        return model.x_t[(m,g), d, t] == 0
    return Constraint.Skip
model.HC6_dias_validos_teo = Constraint(
    model.MG, model.D, model.T, rule=dias_validos_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-7: Solo asignar en días PERMITIDOS — sesiones de cómputo
# ─────────────────────────────────────────────────────────────────────────────
def dias_validos_comp_rule(model, m, g, d, t):
    if d not in dias_permitidos[m]:
        return model.x_c[(m,g), d, t] == 0
    return Constraint.Skip
model.HC7_dias_validos_comp = Constraint(
    model.MG, model.D, model.T, rule=dias_validos_comp_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-8: Materias 100% teóricas → x_c = 0 (no ocupan sala de cómputo)
# ─────────────────────────────────────────────────────────────────────────────
def solo_teo_rule(model, m, g, d, t):
    if tipo_sesion[m] == "teorico":
        return model.x_c[(m,g), d, t] == 0
    return Constraint.Skip
model.HC8_solo_teo = Constraint(model.MG, model.D, model.T, rule=solo_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-9: Materias 100% de cómputo → x_t = 0 (no ocupan salón teórico)
# ─────────────────────────────────────────────────────────────────────────────
def solo_comp_rule(model, m, g, d, t):
    if tipo_sesion[m] == "computo":
        return model.x_t[(m,g), d, t] == 0
    return Constraint.Skip
model.HC9_solo_comp = Constraint(model.MG, model.D, model.T, rule=solo_comp_rule)

print("Restricciones DURAS (HC1–HC9) creadas")


Restricciones DURAS (HC1–HC9) creadas


## 8. Restricciones (Constraints)

Estas restricciones son **deseables** pero no obligatorias.  
Se pueden **desactivar** si causan infactibilidad o si el modelo no converge.

### SC-10 y SC-11: Franja fija

**Objetivo**: que todas las sesiones de un grupo (teóricas y de cómputo) ocurran  
**a la misma hora** en diferentes días.

Por ejemplo, Cálculo grupo 0 con 2 sesiones → idealmente siempre de 9-11  
(martes 9-11 y jueves 9-11), no martes 7-9 y jueves 11-13.

**¿Por qué puede causar problemas?**  
Si hay pocas salas de cómputo, es posible que la única franja disponible para cómputo  
sea diferente a la franja disponible para la sesión teórica.  
Al forzar la misma franja, el modelo puede volverse **infactible**.

**Solución**: se mantienen desactivadas por defecto. Se activan cuando hay capacidad suficiente.


In [66]:
# ─────────────────────────────────────────────────────────────────────────────
# SC-10: Cada grupo elige exactamente UNA franja fija (solo para >1 sesión)
#
#   Σ_t z[m,g,t] = 1    ∀ (m,g) con sesiones_materia[m] > 1
# ─────────────────────────────────────────────────────────────────────────────
def una_franja_rule(model, m, g):
    if sesiones_materia[m] > 1:
        return sum(model.z[(m,g), t] for t in model.T) == 1
    return Constraint.Skip
model.SC10_una_franja = Constraint(model.MG, rule=una_franja_rule)

# ─────────────────────────────────────────────────────────────────────────────
# SC-11: Todas las sesiones de un grupo van en esa franja fija
#
#   Σ_d x_t[m,g,d,t] + Σ_d x_c[m,g,d,t] = sesiones_materia[m] * z[m,g,t]
#   ∀ (m,g) con sesiones_materia[m] > 1, ∀ t
#
# Lectura: si z[m,g,t]=1 (la franja elegida es t), entonces todas las
# sesiones de ese grupo deben estar en esa franja t.
# Si z[m,g,t]=0 → no puede haber ninguna sesión a esa hora.
# ─────────────────────────────────────────────────────────────────────────────
def franja_fija_rule(model, m, g, t):
    if sesiones_materia[m] > 1:
        return (
            sum(model.x_t[(m,g), d, t] for d in dias_permitidos[m]) +
            sum(model.x_c[(m,g), d, t] for d in dias_permitidos[m])
        ) == sesiones_materia[m] * model.z[(m,g), t]
    return Constraint.Skip
model.SC11_franja_fija = Constraint(model.MG, model.T, rule=franja_fija_rule)

# ─────────────────────────────────────────────────────────────────────────────
#
# Por defecto DESACTIVADAS porque con capacidad ajustada pueden causar
# infactibilidad. Actívalas cuando el modelo converja sin ellas.
# Para activar: model.SC10_una_franja.activate()
#               model.SC11_franja_fija.activate()
# ─────────────────────────────────────────────────────────────────────────────
model.SC10_una_franja.deactivate()
model.SC11_franja_fija.deactivate()

print("Restricciones BLANDAS (SC10–SC11) creadas")
print("   Estado SC10 (franja única):  DESACTIVADA")
print("   Estado SC11 (franja fija):   DESACTIVADA")
print()
print("Para activarlas:")
print("   model.SC10_una_franja.activate()")
print("   model.SC11_franja_fija.activate()")


(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
Restricciones BLANDAS (SC10–SC11) creadas
   Estado SC10 (franja única):  DESACTIVADA
   Estado SC11 (franja fija):   DESACTIVADA

Para activarlas:
   model.SC10_una_franja.activate()
   model.SC11_franja_fija.activate()


## 9. Función objetivo

**Maximizar** el número de sesiones asignadas, **penalizando** las que no se pudieron asignar.

```
max  Σ x_t[m,g,d,t] + Σ x_c[m,g,d,t]  — PENALIZACION × Σ (holgura_teo + holgura_comp)
```

- El primer bloque premia cada sesión que sí se asigna.
- El segundo bloque penaliza cada sesión que quedó sin asignar.
- Con `PENALIZACION = 10`, el solver prefiere **mucho** asignar una sesión  
  antes que dejarla en holgura.


In [59]:
PENALIZACION = 10  # Cuánto "cuesta" no asignar una sesión

model.obj = Objective(
    expr=(
        # Premio: sesiones teóricas asignadas
        sum(model.x_t[(m,g), d, t]
            for (m,g) in model.MG for d in model.D for t in model.T)
        # Premio: sesiones de cómputo asignadas
        + sum(model.x_c[(m,g), d, t]
              for (m,g) in model.MG for d in model.D for t in model.T)
        # Penalización: sesiones no asignadas
        - PENALIZACION * sum(
            model.holgura_teo[(m,g)] + model.holgura_comp[(m,g)]
            for (m,g) in model.MG
        )
    ),
    sense=maximize
)

print(f"Función objetivo creada (PENALIZACION = {PENALIZACION})")


Función objetivo creada (PENALIZACION = 10)


## 10. Ejecución del solver

GLPK resolverá el MILP y buscará la asignación óptima.  
El parámetro `tee=False` oculta el log del solver (cámbialo a `True` para ver el detalle).


In [60]:
from pyomo.environ import SolverFactory

solver = SolverFactory(
    'glpk',
    executable='/usr/local/bin/glpsol'
)

print(solver.available())

True


In [61]:
solver = SolverFactory(
    'glpk',
    executable='/usr/local/bin/glpsol'
)

results = solver.solve(model, tee=False)

print("=" * 55)
print("RESULTADO DEL SOLVER")
print("=" * 55)
print(f"   Estado:              {results.solver.status}")
print(f"   Condición de parada: {results.solver.termination_condition}")


RESULTADO DEL SOLVER
   Estado:              ok
   Condición de parada: optimal


## 11. Resultados y diagnóstico de asignación

Aquí se construyen dos tablas:
- **Horario generado**: cada sesión asignada con día, hora y tipo de salón.
- **Resumen por grupo**: si logró asignación completa y qué faltó si no.


In [ ]:
if (results.solver.status == SolverStatus.ok and
        results.solver.termination_condition == TerminationCondition.optimal):

    print("\nSolución óptima encontrada\n")

    # ── Construir tabla de horario ────────────────────────────────────────
    filas_horario = []
    for (m, g) in model.MG:
        for d in model.D:
            for t in model.T:
                try:
                    if value(model.x_t[(m,g), d, t]) >= 0.5:
                        filas_horario.append({
                            "Materia": m, "Grupo": g,
                            "Tipo_Salon": "teorico", "Dia": d, "Hora": t
                        })
                except: pass
                try:
                    if value(model.x_c[(m,g), d, t]) >= 0.5:
                        filas_horario.append({
                            "Materia": m, "Grupo": g,
                            "Tipo_Salon": "computo", "Dia": d, "Hora": t
                        })
                except: pass

    horario_df = pd.DataFrame(filas_horario)

    # ── Construir resumen por grupo ───────────────────────────────────────
    resumen_grupos = []
    for (m, g) in model.MG:
        ht = round(value(model.holgura_teo[(m,g)]))
        hc = round(value(model.holgura_comp[(m,g)]))
        completo = (ht == 0 and hc == 0)

        # Identificar qué días se asignaron y qué faltó
        dias_asig_teo  = []
        dias_asig_comp = []
        for d in dias_permitidos[m]:
            for t in model.T:
                try:
                    if value(model.x_t[(m,g), d, t]) >= 0.5:
                        dias_asig_teo.append(f"{d} {t}")
                except: pass
                try:
                    if value(model.x_c[(m,g), d, t]) >= 0.5:
                        dias_asig_comp.append(f"{d} {t}")
                except: pass

        # Diagnóstico de sesiones faltantes
        mensajes = []
        if ht > 0:
            mensajes.append(
                f"Faltan {ht} sesión(es) teórica(s) — "
                f"posible causa: sin salones teóricos disponibles en "
                f"{', '.join(dias_permitidos[m])}"
            )
        if hc > 0:
            mensajes.append(
                f"Faltan {hc} sesión(es) de cómputo — "
                f"posible causa: salas de cómputo ocupadas en todas las franjas de "
                f"{', '.join(dias_permitidos[m])}"
            )

        resumen_grupos.append({
            "Materia":                    m,
            "Grupo":                      g,
            "Ses. requeridas":            sesiones_materia[m],
            "Ses. asignadas":             sesiones_materia[m] - ht - hc,
            "Logró asignación completa":  "Sí" if completo else "No",
            "Qué falta / diagnóstico":    " | ".join(mensajes) if mensajes else "—",
            "Sesiones teo asignadas":     ", ".join(dias_asig_teo)  or "—",
            "Sesiones comp asignadas":    ", ".join(dias_asig_comp) or "—",
        })

    resumen_df = pd.DataFrame(resumen_grupos)

    # ── Imprimir horario ordenado ─────────────────────────────────────────
    if not horario_df.empty:
        orden_dias  = {d: i for i, d in enumerate(dias)}
        orden_horas = {h: i for i, h in enumerate(horas)}
        horario_df["ord_dia"]  = horario_df["Dia"].map(orden_dias)
        horario_df["ord_hora"] = horario_df["Hora"].map(orden_horas)
        horario_df = (horario_df
                      .sort_values(["Materia","Grupo","ord_dia","ord_hora"])
                      .drop(columns=["ord_dia","ord_hora"]))

        print("=" * 80)
        print("HORARIO GENERADO")
        print("=" * 80)
        print(horario_df.to_string(index=False))
    else:
        print("No se generó ninguna asignación.")

    # ── Imprimir resumen ──────────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("RESUMEN POR GRUPO")
    print("=" * 80)
    cols_resumen = ["Materia","Grupo","Ses. requeridas","Ses. asignadas",
                    "Logró asignación completa","Qué falta / diagnóstico"]
    print(resumen_df[cols_resumen].to_string(index=False))

    # ── Estadísticas finales ──────────────────────────────────────────────
    total       = len(resumen_df)
    completos   = (resumen_df["Logró asignación completa"] == "Sí").sum()
    incompletos = total - completos

    print(f"\n{'='*55}")
    print(f"RESUMEN GENERAL")
    print(f"{'='*55}")
    print(f"   Grupos totales:            {total}")
    print(f"   Asignaciones completas:    {completos}  ({100*completos//total}%)")
    print(f"   Asignaciones incompletas:  {incompletos}")

    if incompletos > 0:
        print("\n DIAGNÓSTICO DE GRUPOS INCOMPLETOS:")
else:
    # ── El solver no encontró solución ────────────────────────────────────
    print("\n EL SOLVER NO ENCONTRÓ SOLUCIÓN")
    print(f"   Estado:    {results.solver.status}")
    print(f"   Condición: {results.solver.termination_condition}")
    print()
    print("Posibles causas:")
    print("   1. Restricciones contradictorias entre sí")
    print("   2. Capacidad total de salones insuficiente para la demanda")
    print("   3. Datos incorrectos (revisar la celda de diagnóstico)")
    print()
    print("Pasos para depurar:")
    print("   → Ejecuta el diagnóstico de capacidad (celda 4)")
    print("   → Desactiva HC3 y HC4 para verificar si el modelo es factible sin límite de salones:")
    print("      model.HC3_cap_teo.deactivate()")
    print("      model.HC4_cap_comp.deactivate()")
    print("   → Si con eso funciona: el problema es de capacidad física")
    print("   → Si sigue fallando: revisa los datos de entrada")


# ── Uso de salones por día y franja ──────────────────────────────────
print("\n" + "=" * 80)
print("USO DE SALONES")
print("=" * 80)

for d in dias:
    for t in horas:
        usados_teo  = sum(
            1 for (m,g) in MG_teo
            if value(model.x_t[(m,g), d, t]) >= 0.5
        )
        usados_comp = sum(
            1 for (m,g) in MG_comp
            if value(model.x_c[(m,g), d, t]) >= 0.5
        )
        print(
            f"   {d:10s} {t}  |  "
            f"Teórico:  {usados_teo}/{slots_teorico[d]}  |  "
            f"Cómputo:  {usados_comp}/{slots_computo[d]}"
        )


✅ Solución óptima encontrada

HORARIO GENERADO
                                Materia  Grupo Tipo_Salon       Dia  Hora
                         Bases de Datos      0    computo     Lunes  9-11
                         Bases de Datos      0    computo Miercoles 11-13
                         Bases de Datos      0    computo   Viernes 11-13
                         Bases de Datos      1    computo     Lunes  9-11
                         Bases de Datos      1    computo Miercoles 11-13
                         Bases de Datos      1    computo   Viernes 11-13
                         Bases de Datos      2    computo     Lunes  9-11
                         Bases de Datos      2    computo Miercoles 11-13
                         Bases de Datos      2    computo   Viernes 14-16
                         Bases de Datos      3    computo     Lunes  9-11
                         Bases de Datos      3    computo Miercoles 11-13
                         Bases de Datos      3    computo   Vier

## 12. Guía de activación/desactivación de restricciones

Esta celda es una **referencia rápida** para experimentar con el modelo.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GUÍA RÁPIDA — Activar y desactivar restricciones
# ═══════════════════════════════════════════════════════════════════
#
# RESTRICCIONES DURAS (desactivar solo para diagnóstico):
# ───────────────────────────────────────────────────────
# model.HC1_ses_teo.deactivate()      # No exigir sesiones teóricas
# model.HC2_ses_comp.deactivate()     # No exigir sesiones de cómputo
# model.HC3_cap_teo.deactivate()      # Ignorar límite de salones teóricos
# model.HC4_cap_comp.deactivate()     # Ignorar límite de salas de cómputo
# model.HC5_una_clase_dia.deactivate()# Permitir >1 clase por día
# model.HC6_dias_validos_teo.deactivate() # Ignorar esquema LMiVi / MaJ (teórico)
# model.HC7_dias_validos_comp.deactivate()# Ignorar esquema LMiVi / MaJ (cómputo)
# model.HC8_solo_teo.deactivate()     # Permitir que una mat. teórica use cómputo
# model.HC9_solo_comp.deactivate()    # Permitir que una mat. cómputo use teórico
#
# RESTRICCIONES BLANDAS (desactivadas por defecto):
# ─────────────────────────────────────────────────
# model.SC10_una_franja.activate()    # Activar: cada grupo elige UNA franja
# model.SC11_franja_fija.activate()   # Activar: todas las sesiones a esa hora
#
# ⚠️  SC10 y SC11 deben activarse JUNTAS o ambas desactivadas.
#     Activar solo una puede generar restricciones contradictorias.
#
# PARA REACTIVAR UNA RESTRICCIÓN DURA:
# model.HC3_cap_teo.activate()
# model.HC4_cap_comp.activate()
# ═══════════════════════════════════════════════════════════════════

print("Este bloque es solo de referencia — no ejecuta nada.")
print("Copia las líneas que necesites a las celdas anteriores.")
